# Quasi-BRDF drift workflow — real data, Libya 4 patch `10_7_1`

Real-data parallel of `BRDF_python_implementation.ipynb`. That notebook validated
the joint quasi-BRDF + drift model on synthetic multi-site data; here we run the
**single-site** version on one real patch, for one channel, for both products.

One patch = one site, so the state vector is just **three parameters**:

$$\mathbf{x} = [\alpha,\ \beta,\ \gamma], \qquad \hat\rho = (\alpha + \beta\,\cos\theta)\,(1 + \gamma t)$$

`α` = brightness baseline, `β` = directional (BRDF) sensitivity to `cos(SZA)`,
`γ` = drift rate (fraction/yr). Unlike matched pairs, this uses **every**
QC-surviving scene (no geometry-matching restriction) — the full-pool payoff.

**Scope:** self-contained (no ground truth, no matched-pairs cross-check),
one channel (default 858 nm, editable), rad and rfl fit on the *same* scene set.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares


def find_repo_root(start=None):
    """Walk up from the notebook's cwd until the emit_pics/ package is found."""
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'emit_pics').is_dir():
            return cand
    raise RuntimeError('Could not locate repo root (no emit_pics/ upward from cwd).')


REPO = find_repo_root()
sys.path.insert(0, str(REPO))

from emit_pics.qc_gate import apply_qc_filter, QCReviewRequired
from emit_pics.delta_calculations import load_scene_spectrum

# ---------------- USER-CONFIGURABLE INPUTS ----------------
PICS_ID           = 'Libya_4'
PATCH_ID          = '10_7_1'
TARGET_WL         = 858.0     # nm; nearest channel is used (editable)
REQUIRE_QC_REVIEW = True      # require a reviewed QC worksheet before fitting
N_BOOTSTRAP       = 1000      # resamples for the gamma uncertainty

DATA      = REPO / 'data'
GEOM_PATH = DATA / 'geometry' / f'geometry_{PICS_ID}_{PATCH_ID}.csv'
QC_PATH   = DATA / 'qc' / f'{PICS_ID}_{PATCH_ID}_qc.csv'
RFL_DIR   = DATA / 'reflectance'
RAD_DIR   = DATA / 'radiance'

PRODUCTS = {'rad': '#1f77b4', 'rfl': '#2ca02c'}   # product -> plot color
print('repo root:', REPO)

## 1. Assemble the per-scene observation table

Three sources join per scene: the **geometry table** (`t`, `cos(SZA)`, `has_rfl`),
the **QC worksheet** (drop rejected scenes), and each scene's **per-channel stats**
(median ρ at the chosen band). To keep rad and rfl on an identical scene set with
equal `n`, we keep only QC-surviving scenes that also have reflectance
(`has_rfl = True`; radiance always exists). Scenes are aligned by **band order**
(the same channel row across acquisitions), and `t` is years since the first
surviving scene.

In [ ]:
if not GEOM_PATH.exists():
    raise FileNotFoundError(f'Missing geometry table {GEOM_PATH}\n'
                            '  -> run the cube extraction for this patch first.')

geom = pd.read_csv(GEOM_PATH)
geom['datetime'] = pd.to_datetime(geom['datetime'])
geom['has_rfl'] = geom['has_rfl'].astype(str).str.strip().str.lower().isin(['true', '1'])
geom = geom.sort_values('datetime').reset_index(drop=True)
n_all = len(geom)

# QC gate (drop keep=FALSE; refuse unless reviewed).
try:
    geom, qc = apply_qc_filter(geom, QC_PATH, require_review=REQUIRE_QC_REVIEW)
except QCReviewRequired as exc:
    raise SystemExit(f'QC gate blocked: {exc}')
n_qc = len(geom)

# Equal-n across products: keep only scenes carrying reflectance.
geom = geom[geom['has_rfl']].reset_index(drop=True)
print(f'scenes: {n_all} total -> {n_qc} after QC -> {len(geom)} with reflectance')

# Reference band grid -> the band index nearest TARGET_WL (band-order alignment).
ref = load_scene_spectrum(geom.loc[0, 'scene_id'], PICS_ID, PATCH_ID, RFL_DIR, RAD_DIR)
if ref is None:
    raise SystemExit('First surviving scene has no radiance CSV; cannot set the band grid.')
BIDX = int((ref['central_wl'] - TARGET_WL).abs().idxmin())
WL_USED = float(ref.loc[BIDX, 'central_wl'])

# Per-scene median rho at that band, for both products (same band row).
recs = []
for _, s in geom.iterrows():
    spec = load_scene_spectrum(s['scene_id'], PICS_ID, PATCH_ID, RFL_DIR, RAD_DIR)
    if spec is None or len(spec) != len(ref):
        print(f"  skip {s['timestamp']}: missing/'inconsistent band grid")
        continue
    recs.append({'timestamp': s['timestamp'], 'datetime': s['datetime'],
                 'sza_mean': s['sza_mean'],
                 'rho_rad': spec.loc[BIDX, 'median_rad'],
                 'rho_rfl': spec.loc[BIDX, 'median_rfl']})

obs = pd.DataFrame(recs).sort_values('datetime').reset_index(drop=True)
obs['t'] = (obs['datetime'] - obs['datetime'].min()).dt.total_seconds() / (365.25 * 86400)
obs['g'] = np.cos(np.radians(obs['sza_mean']))

if len(obs) < 4:
    print(f'WARNING: only {len(obs)} scene(s) — a 3-parameter fit needs > 3 points.')
print(f'\nChannel: band {BIDX} = {WL_USED:.1f} nm (target {TARGET_WL:.0f} nm)')
print(f'n = {len(obs)} scenes | {obs["datetime"].min():%Y-%m-%d} .. {obs["datetime"].max():%Y-%m-%d}')
print(f'SZA {obs["sza_mean"].min():.1f}-{obs["sza_mean"].max():.1f} deg | '
      f'cos(SZA) {obs["g"].min():.3f}-{obs["g"].max():.3f} | t 0-{obs["t"].max():.2f} yr')
obs.head()

## 2. Diagnostic scatters + confounding check (before fitting)

With only three parameters from one site, the fit is trustworthy **only if** there
is real leverage: spread in `cos(SZA)` (to pin `β`), spread in `t` (to pin `γ`),
and — critically — those two not moving together. The two panels below plot ρ vs
each; the point color is the *other* variable, so a color gradient running along
the x-axis is the visual warning of `β`/`γ` confounding. The printed
`corr(cos SZA, t)` quantifies it: near 0 = well separated, near ±1 = confounded.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, (prod, color) in enumerate(PRODUCTS.items()):
    y = obs[f'rho_{prod}']
    s0 = axes[i, 0].scatter(obs['t'], y, c=obs['g'], cmap='viridis',
                            s=32, edgecolors='k', linewidths=0.3)
    axes[i, 0].set_xlabel('t (yr)'); axes[i, 0].set_ylabel(f'{prod} rho (median)')
    fig.colorbar(s0, ax=axes[i, 0], label='cos(SZA)')
    s1 = axes[i, 1].scatter(obs['g'], y, c=obs['t'], cmap='plasma',
                            s=32, edgecolors='k', linewidths=0.3)
    axes[i, 1].set_xlabel('cos(SZA)'); axes[i, 1].set_ylabel(f'{prod} rho (median)')
    fig.colorbar(s1, ax=axes[i, 1], label='t (yr)')
axes[0, 0].set_title('rho vs time  (color = cos SZA)')
axes[0, 1].set_title('rho vs cos(SZA)  (color = time)')
fig.suptitle(f'{PICS_ID} {PATCH_ID} — band {BIDX} ({WL_USED:.0f} nm): leverage check',
             fontsize=12)
fig.tight_layout()
plt.show()

corr_gt = float(np.corrcoef(obs['g'], obs['t'])[0, 1])
print(f'Leverage: cos(SZA) span {obs["g"].max() - obs["g"].min():.3f}, '
      f't span {obs["t"].max():.2f} yr, corr(cos SZA, t) = {corr_gt:+.2f}')
print('  |corr| near 0 -> beta and gamma well separated; near +/-1 -> confounded '
      '(single-patch gamma unreliable, widen to the full PICS).')

## 3. Fit the three-parameter model

`(α + β·cosθ)(1 + γ·t)` by nonlinear least squares, warm-started from an OLS of ρ
vs `cos(SZA)` (giving `α₀`, `β₀`) with `γ₀ = 0` — the same recipe as the synthetic
notebook, minus the multi-site stacking. Fit independently for rad and rfl.

In [ ]:
def fit_site(g, t, rho):
    """Fit [alpha, beta, gamma] for one site/channel; returns a result dict."""
    g = np.asarray(g, float); t = np.asarray(t, float); rho = np.asarray(rho, float)
    ok = np.isfinite(g) & np.isfinite(t) & np.isfinite(rho)
    g, t, rho = g[ok], t[ok], rho[ok]
    a0, b0 = np.linalg.lstsq(np.column_stack([np.ones(g.size), g]), rho, rcond=None)[0]
    res = least_squares(lambda x: (x[0] + x[1] * g) * (1 + x[2] * t) - rho, [a0, b0, 0.0])
    return {'alpha': res.x[0], 'beta': res.x[1], 'gamma': res.x[2],
            'n': int(g.size), 'success': bool(res.success), 'x0': [a0, b0, 0.0]}


fits = {p: fit_site(obs['g'], obs['t'], obs[f'rho_{p}']) for p in PRODUCTS}

print(f'{"":<10}{"alpha":>12}{"beta":>12}{"gamma":>12}{"gamma %/yr":>13}{"n":>5}')
for p, r in fits.items():
    print(f'{p:<10}{r["alpha"]:>12.5f}{r["beta"]:>12.5f}{r["gamma"]:>12.6f}'
          f'{100 * r["gamma"]:>+13.3f}{r["n"]:>5}')

## 4. Quasi-BRDF correction — divide out the site factor

Dividing each scene by its fitted directional factor `(α + β·cosθ)` removes the
geometry dependence and leaves the shared drift `(1 + γ·t)`. If the model fits,
the corrected points collapse onto the `1 + γt` line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharex=True)
tg = np.linspace(float(obs['t'].min()), float(obs['t'].max()), 50)
for ax, (prod, color) in zip(axes, PRODUCTS.items()):
    r = fits[prod]
    site_factor = r['alpha'] + r['beta'] * obs['g']
    ax.scatter(obs['t'], obs[f'rho_{prod}'] / site_factor, s=30, c=color,
               edgecolors='k', linewidths=0.3, zorder=3)
    ax.plot(tg, 1 + r['gamma'] * tg, 'k-', lw=1.7, zorder=4,
            label=f"1 + gamma*t  (gamma = {100 * r['gamma']:+.3f} %/yr)")
    ax.axhline(1.0, color='k', lw=0.6, ls=':')
    ax.set_title(f'{prod.upper()} — corrected'); ax.set_xlabel('t (yr)')
    ax.legend(fontsize=9)
axes[0].set_ylabel(r'$\rho\,/\,(\alpha + \beta\,\cos\theta)$')
fig.suptitle(f'{PICS_ID} {PATCH_ID} — band {BIDX} ({WL_USED:.0f} nm): '
             'site factor divided out', fontsize=12)
fig.tight_layout()
plt.show()

## 5. Bootstrap the drift rate

Resample the scenes with replacement, refit the 3-parameter model, and keep γ.
The spread of the refit γ values is the bootstrap SE; the 2.5/97.5 percentiles are
the 95% CI.

In [ ]:
def bootstrap_gamma(g, t, rho, n_boot, seed=0):
    g = np.asarray(g, float); t = np.asarray(t, float); rho = np.asarray(rho, float)
    ok = np.isfinite(g) & np.isfinite(t) & np.isfinite(rho)
    g, t, rho = g[ok], t[ok], rho[ok]
    base = fit_site(g, t, rho)
    x0 = [base['alpha'], base['beta'], base['gamma']]
    rng = np.random.default_rng(seed)
    n = g.size
    draws = np.empty(n_boot)
    for k in range(n_boot):
        idx = rng.integers(0, n, n)
        res = least_squares(
            lambda x: (x[0] + x[1] * g[idx]) * (1 + x[2] * t[idx]) - rho[idx], x0)
        draws[k] = res.x[2]
    return {'gamma': base['gamma'], 'se': float(draws.std()),
            'ci': np.percentile(draws, [2.5, 97.5]), 'draws': draws}


boots = {p: bootstrap_gamma(obs['g'], obs['t'], obs[f'rho_{p}'], N_BOOTSTRAP, seed=42)
         for p in PRODUCTS}

print(f'{"":<10}{"gamma %/yr":>12}{"boot SE":>10}{"95% CI (%/yr)":>22}')
for p, b in boots.items():
    lo, hi = 100 * b['ci']
    print(f'{p:<10}{100 * b["gamma"]:>+12.3f}{100 * b["se"]:>10.3f}'
          f'{f"[{lo:+.3f}, {hi:+.3f}]":>22}')

fig, ax = plt.subplots(figsize=(7.5, 3.4))
for p, color in PRODUCTS.items():
    ax.hist(100 * boots[p]['draws'], bins=40, color=color, alpha=0.55, label=p)
    ax.axvline(100 * fits[p]['gamma'], color=color, lw=1.6)
ax.axvline(0.0, color='k', lw=0.8, ls='--')
ax.set_xlabel('gamma (%/yr)'); ax.set_ylabel('bootstrap count')
ax.set_title(f'{PICS_ID} {PATCH_ID} — band {BIDX} ({WL_USED:.0f} nm): drift bootstrap')
ax.legend()
fig.tight_layout()
plt.show()

### Reading this / caveats

* **γ** is this patch's single-channel drift estimate (%/yr) at the chosen band,
  from the full scene pool — no geometry-matching restriction. A 95% CI that
  straddles 0 means no significant drift detected at this channel for this patch.
* **Trust it only if the leverage check (step 2) was clean** — enough SZA and time
  spread and low `corr(cos SZA, t)`. A single patch can be geometry-limited; that
  is exactly the motivation to pool the full Libya 4 PICS (more scenes, wider
  geometry) and, eventually, multiple sites with a shared γ.
* **rad vs rfl** share the identical scene set here, so a difference between their
  γ is a product effect (atmosphere / ISOFIT), not different sampling.
* **Next steps:** sweep all channels into a γ(λ) drift spectrum (mask the same
  water-vapor bands for both products), then scale to the full PICS / multi-site
  pooled fit. Change `TARGET_WL` above to inspect a different channel.